In [13]:
# FST and CG3 setup
from fst_runtime.fst import Fst
from cg3_process import disambiguate
from dependency_parsing import cg3_to_conllu_batch

In [14]:
import os
os.environ["PATH"] += os.pathsep + "/usr/local/bin"


In [15]:
import jinja2, rich
from pathlib import Path
from cg3_process import disambiguate, ojibwe_sentence_to_cg3_format  

OJIBWE_SENTENCES_PATH  = "../data/parallel_data/treebank_sentences/ojibwe_gaa.txt"
ENGLISH_SENTENCES_PATH = "../data/parallel_data/treebank_sentences/english_gaa.txt"
CG3_GRAMMAR_PATH       = "../data/CG3_rules/Ojibwe_disambiguation.cg3"
OUT_HTML_PATH          = "../data/disambiguation_reference/gaa_disambiguation.html"
HTML_TITLE             = "Gaa Disambiguation Sample July 20"
FST                    = Fst("../data/fst/ojibwe.att")

def load_lines(path: Path) -> list[str]:
    return [ln.rstrip("\n") for ln in path.read_text(encoding="utf8").splitlines()
            if ln.strip()]

ojibwe  = load_lines(Path(OJIBWE_SENTENCES_PATH))
english = load_lines(Path(ENGLISH_SENTENCES_PATH))
assert len(ojibwe) == len(english), "Parallel files not aligned!"

# Process each sentence
rows = []
for idx, (oj, en) in enumerate(zip(ojibwe, english), 1):
    before = ojibwe_sentence_to_cg3_format(oj, FST)
    after = disambiguate(oj, CG3_GRAMMAR_PATH, FST)
    rows.append({"no": idx, "oj": oj, "en": en,
                 "before": before, "after": after})

# Render HTML using Jinja2
tpl = jinja2.Template(r"""
<!doctype html><html lang="en"><head>
<meta charset="utf-8">
<title>{{ title }} ({{ rows|length }} sentences)</title>
<style>
body{font-family:system-ui,Arial,sans-serif;margin:2rem;}
figure{margin:2rem 0;padding:1rem;border:1px solid #ddd;border-radius:8px;}
figcaption{font-weight:bold;margin-bottom:.5rem;}
.oj{color:#1565c0;} .en{color:#2e7d32;}
pre{background:#f9f9f9;border:1px solid #eee;
    padding:1rem;margin-top:.5rem;
    font:14px/1.4 monospace;white-space:pre-wrap;}
.before{max-height:20em;overflow-y:auto;}
.after {max-height:20em;overflow-y:auto;border-color:#cfd;}
</style></head><body>
<h1>{{ title }} ({{ rows|length }} sentences)</h1>
{% for r in rows %}
<figure>
  <figcaption>#{{ "%02d"|format(r.no) }}
    <span class="oj">{{ r.oj }}</span><br>
    <span class="en">{{ r.en }}</span>
  </figcaption>

  <pre class="before">{{ r.before | e }}</pre>
  <pre class="after">{{ r.after  | e }}</pre>
</figure>
{% endfor %}
</body></html>""")

Path(OUT_HTML_PATH).write_text(
    tpl.render(rows=rows, title=HTML_TITLE), encoding="utf8"
)
rich.print(f"[bold green] Disambiguation booklet written to {OUT_HTML_PATH}")


 Disambiguation booklet written to ../data/disambiguation_reference/gaa_disambiguation.html